# Embeddings

Notebook 1 loaded documents, chunked them, and stored embeddings into `chroma_DB/`.

This notebook picks up from there. The goals are:
- Load the existing ChromaDB vector store (no re-embedding)
- Inspect what's inside: chunk count, a sample embedding vector
- Understand what an embedding actually is before using it for retrieval in notebook 3

In [2]:
# !pip install -r ../requirements.txt

In [3]:
import os
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

In [4]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("API key loaded:", "✅" if OPENAI_API_KEY else "❌ NOT FOUND")

API key loaded: ✅


In [5]:
import sys
sys.path.append(os.path.abspath("../src"))
from vectorstore import load_vectorstore

## Load the existing vector store

Instead of calling `Chroma.from_documents()` (which re-embeds everything), we use `Chroma()` directly and point it at the folder notebook 1 already populated.

We still pass the same embedding model — ChromaDB needs it to embed any *new* queries later, even though the stored chunks are already embedded.

In [9]:
use_OpenAI = True

if use_OpenAI:
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    load_vectorstore("../chroma_DB/", "text-embedding-3-small")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Loaded vectorstore: 102 chunks from '../chroma_DB/'


## Inspect a raw embedding vector

An embedding is just a list of floats — a point in high-dimensional space. Words or chunks with similar *meaning* end up close together in that space. This is what makes semantic search possible.

`text-embedding-3-small` produces vectors with **1536 dimensions**.

In [10]:
if use_OpenAI:
    sample_text = "What does the moth look like?"
    sample_vector = embeddings.embed_query(sample_text)

    print(f"Text: '{sample_text}'")
    print(f"Vector dimensions: {len(sample_vector)}")
    print(f"First 5 values: {sample_vector[:5]}")

Text: 'What does the moth look like?'
Vector dimensions: 1536
First 5 values: [0.049652099609375, -0.016815185546875, -0.04534912109375, 0.0223541259765625, -0.0452880859375]


## Compare two embeddings — similar vs. unrelated

Cosine similarity measures how close two vectors are. 
- `1.0` = identical meaning
- `0.0` = unrelated.

We expect the two similar sentences to score much higher than the unrelated pair.

In [11]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

if use_OpenAI:
    moth_text_A      = "The moth has grey forewings and a buff patch at the apex."
    moth_text_B      = "The forewings are grey with a large prominent buff patch."
    unrelated_text_C = "The stock market closed higher on Friday."

    vec_a = embeddings.embed_query(moth_text_A)
    vec_b = embeddings.embed_query(moth_text_B)
    vec_c = embeddings.embed_query(unrelated_text_C)

    sim_ab = cosine_similarity(vec_a, vec_b)
    sim_ac = cosine_similarity(vec_a, vec_c)
    sim_bc = cosine_similarity(vec_b, vec_c)

    print(f"Similar sentences (A vs B):  {sim_ab:.4f}")
    print(f"Unrelated sentences (A vs C): {sim_ac:.4f}")
    print(f"Unrelated sentences (B vs C): {sim_bc:.4f}")

Similar sentences (A vs B):  0.8186
Unrelated sentences (A vs C): 0.0572
Unrelated sentences (B vs C): 0.0478


## Summary

- The vector store is loaded and ready — **no API calls were made to re-embed**
- Each chunk is stored as a 1536-dimensional vector
- Semantically similar text produces vectors with high cosine similarity

In notebook 3, we query this vector store — a search query gets embedded the same way, and ChromaDB finds the chunks whose vectors are closest to it.